# Lectin ELLA – synovial fluid time series (row layout, one plate per horse)

**Plate layout (from the layout workbook, e.g. `2016.11.11_SF_96well_plate_layouts.xlsx`):**
- Rows A–B: standard curve in duplicate. Column 1 = DPBS blank, columns 2–12 = STD 1–11.
- Rows C–D: LF (replicate 1, 2). Rows E–F: RF (replicate 1, 2). Columns 1–11 = 0 h … 840 h.
- Column 12, rows C–F: SF control 1 / 2 (plate-to-plate check). **These wells are ignored here.**
- Joints/time points with no sample (e.g. Foster (207) RF 504 h, wells E9, F9) are listed in the **MISSING SAMPLES** cell.

**To run:**
1. Fill in the **SETUP** cell (standard concentrations and dilutions for each lectin).
2. Check the **MISSING SAMPLES** cell.
3. **Runtime → Run all**, then select all files at once when the upload button appears.

**Files to upload:**
- The layout workbook. Its file name must match `LAYOUT_FILE` in SETUP (default `layout.xlsx`).
- One plate-reader CSV per lectin per horse, named `<lectin>_<horse>.csv`. The horse can be the name or the ID,
  e.g. `SNA_Gryphon.csv`, `sna_194.csv`, `MAL-II_Foster.csv`.
  Each CSV holds **one** OD block (the reader has already subtracted the 650 nm reference): plate rows A–H in CSV rows 1–8, plate columns 1–12 in CSV columns 1–12. Row 0 and column 0 are headers.

**Outputs:** standard curve plots, `wells_concentrations.csv`, `per_sample_concentrations.csv`, `high_cv_samples.csv`,
`sample_counts.csv`, a violin plot and a time-course plot for each lectin (`violinplot_<lectin>.svg`,
`timecourse_<lectin>.svg`), `model_results_S_vs_C.csv`.

**Sections:** 1 layout · 2 missing samples · 3 plate files · 4 standard curves · 5 well concentrations ·
6 duplicate averaging + CV · 7 sample counts · 8 violin plots · 8b time-course plots · 9 mixed model · download

In [ ]:
import os, re, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter, NullFormatter
from matplotlib.patches import Rectangle
from scipy.optimize import curve_fit
from IPython.display import display
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
from statsmodels.tools.sm_exceptions import ConvergenceWarning
from scipy import stats

plt.rcParams['svg.fonttype'] = 'none'   # editable text in SVGs

## Settings
Edit these two cells before **Run all**.

In [ ]:
#@title SETUP: standards, dilutions, files
#@markdown **One line per lectin** in `LECTINS` below:
#@markdown - `top_std`: STD 1 concentration in µg/mL (the highest standard, column 2)
#@markdown - `step`: dilution between standards (2 = 1:2 serial dilution)
#@markdown - `sample_dilution`: sample dilution as one number (1:1,000 → 1000)
LECTINS = {
    'SNA':    {'top_std': 2.0, 'step': 2, 'sample_dilution': 100000},
    'MAL-I': {'top_std': 1.0, 'step': 2, 'sample_dilution': 1000},
    'AIA':  {'top_std': 1.0, 'step': 2, 'sample_dilution': 100000},
    'LCA':  {'top_std': 1.0, 'step': 2, 'sample_dilution': 1000},
    'PNA':  {'top_std': 5.0, 'step': 4, 'sample_dilution': 100},

}

LAYOUT_FILE = 'layout.xlsx' #@param {type:"string"}

#@markdown **STD 1 is the highest standard** (STD 11 the lowest). Untick if it was the other way round.
STD1_IS_HIGHEST = True #@param {type:"boolean"}

#@markdown **Subtract the DPBS blank** (mean of A1 and B1) from every well on that plate before fitting.
BLANK_SUBTRACT = False #@param {type:"boolean"}

#@markdown Duplicate wells that disagree by more than this CV (%) are flagged.
CV_LIMIT = 20 #@param {type:"number"}

#@markdown Keep values outside the standard range (flagged "extrapolated"). Unticked = left out, as in the lectin notebook.
KEEP_EXTRAPOLATED = False #@param {type:"boolean"}

#@markdown **Where the plate starts in the reader CSV.** Open one CSV: if line 1 is a header (1, 2, 3 … 12) and plate row A is on line 2, leave this at 1. If row A is on line 1 (no header), set 0.
OD_ROW = 1 #@param {type:"integer"}

#@markdown **Untick if the files are already uploaded in this session.**
UPLOAD_FILES = False #@param {type:"boolean"}

if UPLOAD_FILES:
    from google.colab import files
    print('Select ALL files at once: the layout .xlsx and every plate CSV.')
    files.upload()

In [ ]:
#@title MISSING SAMPLES: time points with no sample on the plate
# Each entry is one joint at one time point that was NOT plated.
# Both replicate wells are removed, even if the reader recorded an OD there.
# Counts, plots and the model then use n = 5 instead of 6 for that group.
# Add a line here if another sample is missing in a later run.
MISSING_SAMPLES = [
    {'horse': 'Foster', 'leg': 'RF', 'time_h': 504, 'reason': '0 mL SF remaining (wells E9, F9 empty)'},
    {'horse': 'GG', 'leg': 'RF', 'time_h': 12, 'reason': '0 mL SF remaining (wells E3, F3 empty)' },
    {'horse': 'GG', 'leg': 'LF', 'time_h': 12, 'reason': '0 mL SF remaining (wells C3, D3 empty)' }
]

# Wells whose label contains any of these words are ignored (not curve, not samples, not plots, not model).
IGNORE_LABELS = ['SF control']

display(pd.DataFrame(MISSING_SAMPLES))

## Part 1: Layout and plate files

In [ ]:
# =====================================================================
# SECTION 1: Read the plate maps from the layout workbook
#   One sheet per horse, e.g. 'Gryphon (194)'. The grid starts on the
#   row whose first cell is 'A'; plate columns 1-12 are Excel columns B-M.
#   Every well is classified from its label:
#     'DPBS (blank)'        -> blank
#     'STD 3 (conc. TBD)'   -> standard 3
#     '194 LF C 24h'        -> sample: ID, leg, treatment, time
#     'SF control 1'        -> ignored (IGNORE_LABELS)
#     empty                 -> empty
# =====================================================================
SAMPLE_RE = re.compile(r'^(\d+)\s+([LR][FH])\s+([CS])\s+(\d+)\s*h$', re.I)
STD_RE = re.compile(r'STD\s*(\d+)', re.I)

def classify(label):
    if label is None or (isinstance(label, float) and np.isnan(label)) or str(label).strip() == '':
        return {'type': 'empty'}
    text = ' '.join(str(label).split())          # newlines -> spaces
    if any(w.lower() in text.lower() for w in IGNORE_LABELS):
        return {'type': 'ignored'}
    if 'dpbs' in text.lower() or 'blank' in text.lower():
        return {'type': 'blank'}
    m = STD_RE.search(text)
    if m:
        return {'type': 'standard', 'std_num': int(m.group(1))}
    m = SAMPLE_RE.match(text)
    if m:
        return {'type': 'sample', 'ID': int(m.group(1)), 'Leg': m.group(2).upper(),
                'Tx': m.group(3).upper(), 'time_h': int(m.group(4))}
    return {'type': 'unknown'}


def read_layout(path):
    import openpyxl
    wb = openpyxl.load_workbook(path, data_only=True)
    rows = []
    for ws in wb.worksheets:
        m = re.match(r'\s*(\w+)\s*\((\d+)\)', ws.title)
        if not m:
            continue                                   # 'Legend', 'Well list', ...
        horse, horse_id = m.group(1), int(m.group(2))
        start = next((r for r in range(1, ws.max_row + 1)
                      if str(ws.cell(r, 1).value).strip() == 'A'), None)
        if start is None:
            print(f'WARNING: no plate grid found on sheet {ws.title}')
            continue
        for i, row_letter in enumerate('ABCDEFGH'):
            for col in range(1, 13):
                label = ws.cell(start + i, col + 1).value
                rows.append({'Horse': horse, 'HorseID': horse_id, 'row': row_letter, 'col': col,
                             'label': None if label is None else ' '.join(str(label).split()),
                             **classify(label)})
    return pd.DataFrame(rows)


layout = read_layout(LAYOUT_FILE)
layout['time_h'] = layout['time_h'].astype('Int64')
HORSES = layout[['Horse', 'HorseID']].drop_duplicates().reset_index(drop=True)
print('Horses / plates in the layout:')
display(HORSES)
print('Wells per type on each plate:')
display(pd.crosstab(layout['Horse'], layout['type']))

unknown = layout[layout['type'] == 'unknown']
if len(unknown):
    print('WARNING: these labels were not recognised and are left out:')
    display(unknown[['Horse', 'row', 'col', 'label']])

bad_id = layout[(layout['type'] == 'sample') & (layout['ID'] != layout['HorseID'])]
if len(bad_id):
    print('WARNING: sample ID does not match the sheet name:')
    display(bad_id[['Horse', 'HorseID', 'row', 'col', 'label']])

In [ ]:
# =====================================================================
# SECTION 2: Apply MISSING SAMPLES and check for any other gaps
#   - Wells listed in MISSING SAMPLES are set to 'missing' (removed).
#   - Every joint should have all 11 time points. Any other gap that is
#     not in MISSING SAMPLES is printed so it can be added on purpose.
# =====================================================================
samples = layout['type'] == 'sample'
for m in MISSING_SAMPLES:
    hit = (samples & (layout['Horse'].str.lower() == m['horse'].lower())
           & (layout['Leg'] == m['leg']) & (layout['time_h'] == m['time_h']))
    layout.loc[hit, 'type'] = 'missing'
    print(f"{m['horse']} {m['leg']} {m['time_h']} h: {int(hit.sum())} well(s) in the layout removed "
          f"(0 means the layout already leaves them empty). Reason: {m.get('reason', '')}")

# Expected design: every joint x every time point seen anywhere on the plates
s = layout[layout['type'] == 'sample']
all_times = sorted(s['time_h'].unique())
joints = s[['Horse', 'Leg', 'Tx']].drop_duplicates()
expected = joints.merge(pd.DataFrame({'time_h': all_times}), how='cross')
present = s[['Horse', 'Leg', 'Tx', 'time_h']].drop_duplicates()
gaps = expected.merge(present, how='left', indicator=True)
gaps = gaps[gaps['_merge'] == 'left_only'].drop(columns='_merge')

listed = {(m['horse'].lower(), m['leg'], m['time_h']) for m in MISSING_SAMPLES}
gaps['in_MISSING_SAMPLES'] = [(h.lower(), l, t) in listed
                              for h, l, t in zip(gaps['Horse'], gaps['Leg'], gaps['time_h'])]
print('\nTime points with no sample:')
display(gaps if len(gaps) else 'none')
if (~gaps['in_MISSING_SAMPLES']).any():
    print('WARNING: the gaps marked False are not in MISSING SAMPLES. '
          'Check the layout, or add them to MISSING SAMPLES if they are expected.')

# Replicate number = 1 for the upper row of the pair, 2 for the lower row
s_idx = layout['type'] == 'sample'
layout.loc[s_idx, 'Replicate'] = (layout[s_idx]
    .groupby(['Horse', 'Leg', 'time_h'])['row'].rank(method='first').astype(int))

In [ ]:
# =====================================================================
# SECTION 3: Find the plate-reader files and read them
#   File name = <lectin>_<horse>.csv, horse by name or ID.
#   OD = the reader's corrected value (650 nm already subtracted by the machine).
# =====================================================================
OUTPUT_CSVS = {'wells_concentrations.csv', 'per_sample_concentrations.csv', 'high_cv_samples.csv',
               'sample_counts.csv', 'model_results_S_vs_C.csv'}
name_lookup = {h.lower(): h for h in HORSES['Horse']}
name_lookup.update({str(i): h for h, i in zip(HORSES['Horse'], HORSES['HorseID'])})
lectin_lookup = {k.lower().replace('-', '').replace('_', ''): k for k in LECTINS}

plate_files, not_used = {}, []
for f in sorted(os.listdir('.')):
    if not f.lower().endswith('.csv') or f in OUTPUT_CSVS:
        continue
    tokens = re.split(r'[_\s]+', f[:-4])
    horse = next((name_lookup[t.lower()] for t in tokens if t.lower() in name_lookup), None)
    rest = ''.join(t for t in tokens if t.lower() not in name_lookup)
    lectin = lectin_lookup.get(rest.lower().replace('-', '').replace('_', ''))
    if horse and lectin:
        if (lectin, horse) in plate_files:
            raise ValueError(f'Two files for {lectin} / {horse}: {plate_files[(lectin, horse)]} and {f}')
        plate_files[(lectin, horse)] = f
    else:
        not_used.append(f)

if not plate_files:
    raise FileNotFoundError('No plate files found. Names must be <lectin>_<horse>.csv '
                            f'with the lectin in SETUP ({list(LECTINS)}).')
print('Plate files:')
display(pd.Series(plate_files).rename('file').rename_axis(['lectin', 'horse']).reset_index())
if not_used:
    print('NOT USED (lectin not in SETUP or horse not recognised):', not_used)
for lectin in LECTINS:
    have = {h for (l, h) in plate_files if l == lectin}
    lacking = [h for h in HORSES['Horse'] if h not in have]
    if have and lacking:
        print(f'Note: {lectin} has no file for {lacking}')


def read_plate(path):
    raw = pd.read_csv(path, header=None)
    od = raw.iloc[OD_ROW:OD_ROW + 8, 1:13].apply(pd.to_numeric, errors='coerce').values
    plate = pd.DataFrame(od, index=list('ABCDEFGH'), columns=range(1, 13))
    return plate.rename_axis('row').reset_index().melt(id_vars='row', var_name='col', value_name='OD')


od_long = pd.concat([read_plate(f).assign(lectin=l, Horse=h) for (l, h), f in plate_files.items()],
                    ignore_index=True)
od_long['col'] = od_long['col'].astype(int)

wells = od_long.merge(layout, on=['Horse', 'row', 'col'], how='left')

# Optional blank subtraction (per plate)
blank = wells[wells['type'] == 'blank'].groupby(['lectin', 'Horse'])['OD'].agg(['mean', 'std'])
blank.columns = ['blank_mean', 'blank_sd']
wells = wells.merge(blank.reset_index(), on=['lectin', 'Horse'], how='left')
if BLANK_SUBTRACT:
    wells['OD'] = wells['OD'] - wells['blank_mean']
print('DPBS blank OD per plate (A1, B1):')
display(blank.round(4))

# Sanity check: the missing wells should be empty on the reader too
miss = wells[wells['type'] == 'missing']
if len(miss):
    print('ODs read in the MISSING SAMPLES wells (not used; should be close to the blank):')
    display(miss[['lectin', 'Horse', 'row', 'col', 'OD', 'blank_mean']].round(4))

## Part 2: Standard curves and concentrations

In [ ]:
#@title OPTIONAL: leave a bad standard out of a curve
# Normally leave this empty: {}
# Use it only if one standard point is clearly wrong on one plate, e.g. the
# top standard reads LOWER than the one below it (a "hook"), or a pipetting error.
# Format: {('lectin', 'horse'): [standard numbers to leave out]}
# Example: {('SNA', 'Foster'): [1]}  -> STD 1 is not used for Foster's SNA curve.
# The left-out points still show on that curve's plot as grey x marks.
DROP_STANDARDS = {}

In [ ]:
# =====================================================================
# SECTION 4: 4PL standard curve, one per lectin per horse plate
#   Standards: rows A and B (duplicates), STD 1-11 in columns 2-12.
#   STD n = top_std / step**(n-1)   (STD 1 highest, see SETUP)
#   The DPBS blank is not a curve point; it is drawn as a dotted line.
# =====================================================================
def log4pl(x, A, B, C, D):
    return ((A - D) / (1.0 + (x / C) ** B)) + D

def inv_log4pl(y, A, B, C, D):
    return C * (((A - D) / (y - D)) - 1) ** (1 / B)


def std_conc(lectin, std_num, n_std):
    cfg = LECTINS[lectin]
    k = std_num - 1 if STD1_IS_HIGHEST else n_std - std_num
    return cfg['top_std'] / cfg['step'] ** k


def fit_plate(lectin, horse, plot=True):
    plate = wells[(wells['lectin'] == lectin) & (wells['Horse'] == horse)]
    std = plate[plate['type'] == 'standard'].copy()
    n_std = int(std['std_num'].max())
    std['conc'] = [std_conc(lectin, n, n_std) for n in std['std_num']]
    dropped = DROP_STANDARDS.get((lectin, horse), [])
    used = std[~std['std_num'].isin(dropped)]

    g = used.groupby('conc')['OD'].agg(['mean', 'std']).sort_index()
    x, y = g.index.values, g['mean'].values
    p0 = [y.min(), 1.0, np.exp(np.mean(np.log(x))), y.max()]
    params, _ = curve_fit(log4pl, x, y, p0=p0, maxfev=20000,
                          bounds=([-np.inf, 0.01, 1e-12, -np.inf], [np.inf, 20, np.inf, np.inf]))
    A, B, C, D = params
    yfit = log4pl(x, *params)
    r2 = 1 - np.sum((y - yfit) ** 2) / np.sum((y - y.mean()) ** 2)

    if plot:
        fig, ax = plt.subplots(figsize=(7, 5))
        xs = np.geomspace(x.min(), x.max(), 200)
        ax.plot(xs, log4pl(xs, *params), '-', color='0.15', lw=1.5,
                label=f'4PL fit (R² = {r2:.4f})')
        ax.errorbar(x, y, yerr=g['std'].values, fmt='s', ms=6, color='0.15', mfc='white',
                    mew=1.5, capsize=4, label='Standards (mean ± SD, n = 2)', zorder=3)
        if dropped:
            d = std[std['std_num'].isin(dropped)].groupby('conc')['OD'].mean()
            ax.plot(d.index, d.values, 'x', color='0.5', ms=8, label='Standard left out')
        if not BLANK_SUBTRACT:
            ax.axhline(plate['blank_mean'].iloc[0], ls=':', color='0.5', lw=1, label='DPBS blank')
        ax.set_xscale('log')
        ax.set_xticks(x)
        ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _: f'{v:.3g}'))
        ax.xaxis.set_minor_formatter(NullFormatter())
        ax.tick_params(axis='x', labelrotation=45)
        ax.set_xlabel('Concentration (µg/mL)')
        ax.set_ylabel('OD 450' + (' (blank subtracted)' if BLANK_SUBTRACT else ''))
        ax.set_title(f'{lectin} standard curve: {horse}')
        ax.spines[['top', 'right']].set_visible(False)
        ax.legend(loc='upper left', frameon=False)
        fig.tight_layout()
        plt.show()

    return {'params': dict(A=A, B=B, C=C, D=D), 'lowest': x.min(), 'highest': x.max(),
            'r2': r2, 'n_std': len(x)}


curves = {}
for (lectin, horse) in sorted(plate_files):
    curves[(lectin, horse)] = fit_plate(lectin, horse)

curve_table = pd.DataFrame([{'lectin': l, 'horse': h, **c['params'], 'R2': c['r2'],
                             'lowest_std': c['lowest'], 'highest_std': c['highest'],
                             'standards_used': c['n_std']}
                            for (l, h), c in curves.items()])
display(curve_table.round(4))

In [ ]:
# =====================================================================
# SECTION 5: Every sample well -> µg/mL
#   conc = C * ((A - D) / (OD - D) - 1) ** (1 / B), using its own plate's
#   curve, then x sample_dilution.
#   status: ok            inside the standards
#           extrapolated  outside the standards (kept only if KEEP_EXTRAPOLATED)
#           off curve     past the flat top/bottom of the curve: no value
#   Output: wells_concentrations.csv
# =====================================================================
def od_to_conc(od, curve):
    A, B, C, D = curve['params'].values()
    if not np.isfinite(od) or od >= max(A, D) or od <= min(A, D):
        return np.nan, 'off curve'
    conc = inv_log4pl(od, A, B, C, D)
    if not np.isfinite(conc):
        return np.nan, 'off curve'
    if conc < curve['lowest'] or conc > curve['highest']:
        return conc, 'extrapolated'
    return conc, 'ok'


sw = wells[wells['type'] == 'sample'].copy()
res = [od_to_conc(od, curves[(l, h)]) for od, l, h in zip(sw['OD'], sw['lectin'], sw['Horse'])]
sw['conc_on_plate'] = [r[0] for r in res]
sw['status'] = [r[1] for r in res]
sw['dilution'] = sw['lectin'].map(lambda l: LECTINS[l]['sample_dilution'])
usable = ['ok', 'extrapolated'] if KEEP_EXTRAPOLATED else ['ok']
sw['conc_ug_ml'] = (sw['conc_on_plate'] * sw['dilution']).where(sw['status'].isin(usable))
sw['well'] = sw['row'] + sw['col'].astype(str)

keep_cols = ['lectin', 'Horse', 'HorseID', 'Leg', 'Tx', 'time_h', 'Replicate', 'well',
             'OD', 'status', 'conc_on_plate', 'dilution', 'conc_ug_ml']
sw = sw[keep_cols].sort_values(['lectin', 'Horse', 'Leg', 'time_h', 'Replicate'])
sw.to_csv('wells_concentrations.csv', index=False)
print('Wells per status:')
display(pd.crosstab([sw['lectin'], sw['Horse']], sw['status']))

In [ ]:
# =====================================================================
# SECTION 6: Average the duplicate wells, CV flag, log2
#   One value per lectin x horse x leg x time point.
#   Outputs: per_sample_concentrations.csv, high_cv_samples.csv
# =====================================================================
keys = ['lectin', 'Horse', 'HorseID', 'Leg', 'Tx', 'time_h']
per_sample = (sw.groupby(keys)
                .agg(conc_ug_ml=('conc_ug_ml', 'mean'),
                     n_wells=('conc_ug_ml', 'count'),
                     sd=('conc_ug_ml', 'std'),
                     wells=('well', ', '.join))
                .reset_index())
per_sample['time_h'] = per_sample['time_h'].astype(int)
per_sample['cv_pct'] = 100 * per_sample['sd'] / per_sample['conc_ug_ml']
per_sample['high_cv'] = per_sample['cv_pct'] > CV_LIMIT
per_sample['log2_conc'] = np.log2(per_sample['conc_ug_ml'])
per_sample['Joint'] = per_sample['Horse'] + '_' + per_sample['Leg']
per_sample['Tx_label'] = per_sample['Tx'].map({'C': 'Saline', 'S': 'Synovitis'})
per_sample.drop(columns='sd').to_csv('per_sample_concentrations.csv', index=False)

high_cv = per_sample[per_sample['high_cv']]
high_cv.to_csv('high_cv_samples.csv', index=False)
print(f'{len(high_cv)} sample(s) with duplicate CV > {CV_LIMIT}%:')
display(high_cv[keys + ['wells', 'conc_ug_ml', 'cv_pct']].round(2) if len(high_cv) else 'none')

one_well = per_sample[per_sample['n_wells'] == 1]
none_ = per_sample[per_sample['n_wells'] == 0]
print(f'{len(one_well)} sample(s) rest on 1 usable well; {len(none_)} have no usable well '
      '(both off curve / extrapolated):')
display(pd.concat([one_well, none_])[keys + ['wells', 'n_wells']] if len(one_well) + len(none_) else 'none')

In [ ]:
# =====================================================================
# SECTION 7: How many samples per group (shows the Foster gap)
#   n = number of horses with a usable value at each time point.
#   Output: sample_counts.csv
# =====================================================================
counts = (per_sample.dropna(subset=['conc_ug_ml'])
          .pivot_table(index=['lectin', 'Tx_label'], columns='time_h',
                       values='conc_ug_ml', aggfunc='count', fill_value=0))
counts.to_csv('sample_counts.csv')
display(counts)

## Part 3: Plots

In [ ]:
# =====================================================================
# SECTION 8: Split violin + points, Saline vs Synovitis at each time point
#   Left half / blue circles = Saline (C), right half / orange squares = Synovitis (S).
#   One SVG per lectin: violinplot_<lectin>.svg
#   With 5-6 horses per group the violins are rough; the points carry the data.
# =====================================================================
TX_COLORS = {'C': '#2f6fd0', 'S': '#e07b28'}
TX_MARKER = {'C': 'o', 'S': 's'}
TX_NAME = {'C': 'Saline', 'S': 'Synovitis'}
times = sorted(per_sample['time_h'].unique())
rng = np.random.default_rng(0)

for lectin in [l for l in LECTINS if l in set(per_sample['lectin'])]:
    d = per_sample[(per_sample['lectin'] == lectin)].dropna(subset=['conc_ug_ml'])
    fig, ax = plt.subplots(figsize=(11, 4.5))
    for j, t in enumerate(times):
        for tx, side in (('C', -1), ('S', 1)):
            vals = d[(d['time_h'] == t) & (d['Tx'] == tx)]['conc_ug_ml'].values
            if len(vals) == 0:
                continue
            if len(vals) > 1 and np.ptp(vals) > 0:
                parts = ax.violinplot(vals, positions=[j], widths=0.8, showextrema=False)
                for body in parts['bodies']:
                    body.set_facecolor(TX_COLORS[tx]); body.set_edgecolor('black'); body.set_alpha(0.35)
                    v = body.get_paths()[0].vertices
                    v[:, 0] = np.minimum(v[:, 0], j) if side < 0 else np.maximum(v[:, 0], j)
            ax.scatter(j + side * 0.16 + rng.uniform(-0.05, 0.05, len(vals)), vals,
                       color=TX_COLORS[tx], marker=TX_MARKER[tx], edgecolor='black',
                       s=22, alpha=0.85, zorder=3)
    ax.set_xticks(range(len(times)))
    ax.set_xticklabels([f'{t}' for t in times])
    ax.set_xlim(-0.6, len(times) - 0.4)
    ax.set_xlabel('Time (h)')
    ax.set_yscale('log')
    ax.yaxis.set_major_formatter(FuncFormatter(lambda y, _: f'{y:g}'))
    ax.yaxis.set_minor_formatter(FuncFormatter(lambda y, _: f'{y:g}'))
    ax.set_ylabel('glycoprotein-equivalent (µg/mL)')
    ax.set_title(lectin, fontsize=15, fontweight='bold')
    ax.spines[['top', 'right']].set_visible(False)
    handles = [plt.Line2D([0], [0], marker=TX_MARKER[t], color='none', markerfacecolor=TX_COLORS[t],
                          markeredgecolor='black', markersize=7, label=TX_NAME[t]) for t in ('C', 'S')]
    ax.legend(handles=handles, frameon=False, loc='upper left', bbox_to_anchor=(1.0, 1.0))
    fig.tight_layout()
    fig.savefig(f'violinplot_{lectin}.svg', bbox_inches='tight')
    plt.show(); plt.close(fig)
    print('Saved', f'violinplot_{lectin}.svg')

In [ ]:
# =====================================================================
# SECTION 8b: Time course, Saline vs Synovitis
#   Thin lines = each joint. Thick line = geometric mean per group,
#   shaded band = 95% CI (t-interval on log2 values).
#   x-axis is piecewise linear (X_BREAKS / X_FRAC) with a break mark.
#   One SVG per lectin: timecourse_<lectin>.svg
# =====================================================================
# --- piecewise x-scale --------------------------------------------------
# Breakpoints in hours, and where each one sits along the axis (0 = left, 1 = right).
# Spacing is linear (proportional) inside each segment. Change X_FRAC to give a
# segment more or less room.
X_BREAKS = np.array([0, 72, 672], dtype=float)
X_FRAC   = np.array([0, 0.55, 1.0])

def _piecewise(src, dst):
    """Linear interpolation between breakpoints, extrapolating at both ends
    so axis margins and edge markers don't get clamped."""
    lo = (dst[1] - dst[0]) / (src[1] - src[0])
    hi = (dst[-1] - dst[-2]) / (src[-1] - src[-2])
    def f(v):
        v = np.asarray(v, dtype=float)
        out = np.interp(v, src, dst)
        out = np.where(v < src[0],  dst[0]  + (v - src[0])  * lo, out)
        out = np.where(v > src[-1], dst[-1] + (v - src[-1]) * hi, out)
        return out
    return f

x_fwd = _piecewise(X_BREAKS, X_FRAC)
x_inv = _piecewise(X_FRAC, X_BREAKS)

def mark_breaks(ax, at=72, offset=0.02, half_width=0.008, slant=0.012):
    """Cut a gap through the data and the x-axis just after `at` (hours),
    with // marks on the axis line at both edges of the gap."""
    x0, x1 = ax.get_xlim()
    frac = (x_fwd(at) - x_fwd(x0)) / (x_fwd(x1) - x_fwd(x0))   # position of `at` in axes units
    c = frac + offset                                          # gap centre, just right of the point
    left, right = c - half_width, c + half_width

    # white strip over the lines, shading and axis line
    ax.add_patch(Rectangle((left, -0.03), right - left, 1.06, transform=ax.transAxes,
                           facecolor='white', edgecolor='none', zorder=4, clip_on=False))

    # slanted marks at each edge of the gap
    w, h = ax.figure.get_size_inches() * ax.get_position().size
    dy = slant * w / h                                       # keeps the slant angle even
    for e in (left, right):
        ax.plot([e - slant, e + slant], [-dy, dy], transform=ax.transAxes,
                color='black', lw=0.8, clip_on=False, zorder=5)

# --- plotting -----------------------------------------------------------
for lectin in [l for l in LECTINS if l in set(per_sample['lectin'])]:
    d = per_sample[per_sample['lectin'] == lectin].dropna(subset=['conc_ug_ml'])
    fig, ax = plt.subplots(figsize=(11, 4.5))
    ax.set_xscale('function', functions=(x_fwd, x_inv))

    for joint, g in d.groupby('Joint'):
        g = g.sort_values('time_h')
        ax.plot(g['time_h'], g['conc_ug_ml'], '-', color=TX_COLORS[g['Tx'].iloc[0]],
                lw=0.8, alpha=0.45)
    for tx in ('C', 'S'):
        s = (d[d['Tx'] == tx]
             .groupby('time_h')['log2_conc']
             .agg(['mean', 'std', 'count']))
        tcrit = stats.t.ppf(0.975, s['count'] - 1)          # NaN where n < 2
        half = tcrit * s['std'] / np.sqrt(s['count'])
        gm, lo, hi = 2 ** s['mean'], 2 ** (s['mean'] - half), 2 ** (s['mean'] + half)

        ax.fill_between(s.index, lo, hi, color=TX_COLORS[tx], alpha=0.18, lw=0,
                        zorder=2, label=f'{TX_NAME[tx]} 95% CI')
        ax.plot(s.index, gm, '-', marker=TX_MARKER[tx], color=TX_COLORS[tx],
                lw=2, ms=8, markeredgecolor='white', label=f'{TX_NAME[tx]} (geometric mean)',
                zorder=3, clip_on=False)

    ax.set_xlim(X_BREAKS[0], X_BREAKS[-1])
    ax.set_xticks(times)
    ax.set_xticklabels([f'{t:g}' for t in times])
    ax.xaxis.set_minor_locator(plt.NullLocator())
    mark_breaks(ax, at=72)                   # gap in the lines and axis just after 72 h

    ax.set_xlabel('Time (h)', size=16)
    ax.set_yscale('log')
    ax.yaxis.set_major_formatter(FuncFormatter(lambda y, _: f'{y:g}'))
    ax.yaxis.set_minor_formatter(FuncFormatter(lambda y, _: f'{y:g}'))
    ax.set_ylabel('glycoprotein-equivalent (µg/mL)', size=16)
    ax.set_title(lectin, fontweight='bold', size=20)
    ax.spines[['top', 'right']].set_visible(False)
    ax.legend(frameon=False, loc='upper left', bbox_to_anchor=(1.0, 1.0))
    fig.tight_layout()
    fig.savefig(f'timecourse_{lectin}.svg', bbox_inches='tight')
    plt.show(); plt.close(fig)
    print('Saved', f'timecourse_{lectin}.svg')

## Part 4: Statistics

In [ ]:
# =====================================================================
# SECTION 9: Synovitis vs Saline at each time point, one mixed model per lectin
#
#   log2_conc ~ Tx * Time + (1 | Horse) + (1 | Joint within Horse)
#
#   - Each horse has one Saline joint and one Synovitis joint, so the
#     comparison is within horse.
#   - Joint (horse x leg) is a second random effect because the same
#     joint is sampled at every time point.
#   - Time is categorical (0, 6, ... 840 h); 0 h is the reference.
#   - The model uses whatever samples exist, so Foster's missing
#     504 h Synovitis sample just lowers n at that time point.
#   - Result: S vs C fold change at each time point with 95% CI;
#     BH correction across the time points within each lectin.
#   Output: model_results_S_vs_C.csv
# =====================================================================
def fit_model(sub):
    """Horse + joint random effects; falls back to horse only if that fails."""
    for vc in ({'Joint': '0 + C(Joint)'}, None):
        for method in (None, 'powell', 'lbfgs'):
            try:
                with warnings.catch_warnings():
                    warnings.simplefilter('ignore', ConvergenceWarning)
                    m = smf.mixedlm('log2_conc ~ Tx * C(time_h)', sub, groups=sub['Horse'],
                                    vc_formula=vc)
                    fit = m.fit(reml=True, **({} if method is None else {'method': method}))
                if fit.converged:
                    return fit, 'horse + joint' if vc else 'horse only'
            except (np.linalg.LinAlgError, ValueError):
                continue
    return None, 'failed'


def s_vs_c(fit, t, t0):
    names = list(fit.fe_params.index)
    L = np.zeros((1, len(names)))
    L[0, names.index('Tx[T.S]')] = 1
    inter = f'Tx[T.S]:C(time_h)[T.{t}]'
    if t != t0 and inter in names:
        L[0, names.index(inter)] = 1
    r = fit.t_test(L)
    lo, hi = np.squeeze(r.conf_int())
    return float(np.squeeze(r.effect)), float(lo), float(hi), float(np.squeeze(r.pvalue))


rows = []
for lectin in [l for l in LECTINS if l in set(per_sample['lectin'])]:
    sub = per_sample[per_sample['lectin'] == lectin].dropna(subset=['log2_conc']).copy()
    sub['Tx'] = pd.Categorical(sub['Tx'], ['C', 'S'])
    fit, re_used = fit_model(sub)
    if fit is None:
        print(f'{lectin}: model could not be fitted, skipped.')
        continue
    t0 = min(sub['time_h'])
    for t in sorted(sub['time_h'].unique()):
        est, lo, hi, p = s_vs_c(fit, t, t0)
        rows.append({'lectin': lectin, 'time_h': t,
                     'n_saline': int(((sub['time_h'] == t) & (sub['Tx'] == 'C')).sum()),
                     'n_synovitis': int(((sub['time_h'] == t) & (sub['Tx'] == 'S')).sum()),
                     'log2FC_S_vs_C': est, 'fold_change': 2 ** est,
                     'FC_CI_low': 2 ** lo, 'FC_CI_high': 2 ** hi, 'p_raw': p,
                     'random_effects': re_used})

results = pd.DataFrame(rows)
results['p_adj_BH'] = np.nan
for lectin, idx in results.groupby('lectin').groups.items():
    results.loc[idx, 'p_adj_BH'] = multipletests(results.loc[idx, 'p_raw'], method='fdr_bh')[1]
results['significant_BH'] = results['p_adj_BH'] < 0.05
results.to_csv('model_results_S_vs_C.csv', index=False)

show = results.copy()
for c in ['fold_change', 'FC_CI_low', 'FC_CI_high']:
    show[c] = show[c].round(2)
for c in ['p_raw', 'p_adj_BH']:
    show[c] = show[c].map(lambda p: f'{p:.2e}')
display(show.drop(columns='log2FC_S_vs_C'))
print('fold_change = Synovitis / Saline at that time point (e.g. 1.8 = Synovitis 1.8x higher).')

## Download

In [ ]:
#@title Download all results
DOWNLOAD = True #@param {type:"boolean"}
outputs = [f for f in sorted(os.listdir('.'))
           if f.endswith('.svg') or f in ('wells_concentrations.csv', 'per_sample_concentrations.csv',
                                         'high_cv_samples.csv', 'sample_counts.csv',
                                         'model_results_S_vs_C.csv')]
print(outputs)
if DOWNLOAD:
    try:
        from google.colab import files
        import zipfile
        with zipfile.ZipFile('ELLA_timeseries_results.zip', 'w') as z:
            for f in outputs:
                z.write(f)
        files.download('ELLA_timeseries_results.zip')
    except ImportError:
        print('Not running in Colab: files are in the current folder.')